# Step 16 — the k-means arm, as a comparator

Reads `step12_panels.rds` and `step13_clusters.rds`. Writes `fig15_arms_*.png`.

Step 13 ran two clustering arms and reported the ARI between the free protein clustering and the
WGCNA module labels. Step 14 drew only the hierarchical arm, so the comparison existed as numbers
and not as a picture — which is not a comparison anyone can check.

This notebook draws both, side by side:

- **hierarchical** — cut the ward.D2 / Minkowski dendrogram at 4 groups
- **k-means** — partition both axes directly at k = 4

**Everything else is held fixed.** Same cohorts, same panels, same colour scale, same trait
annotation with the same shared palette, and the same Minkowski distance and ward.D2 linkage
rebuilding the trees inside each slice. The split is the only thing that differs, so any difference
in block structure is the clustering method and nothing else.

The trait legend is `fig14_trait_legend.png` from step 14, built from the same colour maps used
here — fifteen legends do not fit a 2 × 3 grid without clipping.

In [ ]:
suppressMessages({library(ComplexHeatmap); library(circlize); library(grid)})
source("../src/paths.R")
options(stringsAsFactors = FALSE); set.seed(42)
P <- readRDS(art("step12_panels.rds")); K <- readRDS(art("step13_clusters.rds"))
D <- P$D; spec <- P$spec; MINK_P <- P$MINK_P; LINK <- P$LINK; SITES <- P$SITES
RES <- K$RES
W <- lapply(SITES, function(s) readRDS(art("wgcna_%s.rds", s))); names(W) <- SITES

MAX_LABELS <- 80      # a RENDERING limit, not a statistical one

suppressMessages({library(ComplexHeatmap);library(circlize);library(grid)})
dmink <- function(m) dist(m, method="minkowski", p=MINK_P)

# ── shared trait colours ──────────────────────────────────────────────────
# Every panel MUST use the same mapping, or a legend is a lie and the cohorts
# cannot be compared by eye. A fresh rowAnnotation() per panel gets its own
# palette, so the maps are built once here, over the pooled range of all three
# cohorts, and passed to every panel.
#
# Hue carries the antigen/organ system from the `group` column of
# cohorts/clinical-traits.csv, so traits that belong together read together.
GROUP_HUE <- c(activity = "#B2182B", complement = "#2166AC", history = "#5B5B5B",
               haematology = "#1B7837", renal = "#762A83", demographic = "#E08214",
               snRNP = "#B2182B", RoLa = "#B2182B", dsDNA = "#B2182B")
BINARY_COL <- c(Negative = "#F0F0F0", Borderline = "#FDB863", Positive = "#B2182B")

build_trait_colours <- function(spec, metas) {
  out <- list()
  for (i in seq_len(nrow(spec))) {
    src <- spec$source_column[i]
    v   <- unlist(lapply(metas, function(m) m[[src]]))
    hue <- GROUP_HUE[[spec$group[i]]]
    if (spec$type[i] == "numeric") {
      # 5th-95th percentile, not min-max: uPCR runs 0-1286 and a single outlier
      # would flatten every other patient to the same colour.
      qs <- quantile(as.numeric(v), c(.05, .95), na.rm = TRUE)
      if (diff(qs) == 0) qs <- range(as.numeric(v), na.rm = TRUE)
      out[[src]] <- colorRamp2(c(qs[1], qs[2]), c("#FFFFFF", hue))
    } else if (spec$type[i] == "binary") {
      out[[src]] <- BINARY_COL[intersect(names(BINARY_COL), unique(as.character(v)))]
    } else {                                   # ordinal: the age bands, in order
      lv <- sort(unique(as.character(v[!is.na(v) & v != ""])))
      lv <- lv[order(as.numeric(sub("-.*", "", lv)))]
      out[[src]] <- setNames(colorRampPalette(c("#FFFFFF", hue))(length(lv)), lv)
    }
  }
  out
}
TRAIT_COLS <- build_trait_colours(spec, lapply(W, `[[`, "meta"))

# show_leg: the trait legend is drawn on ONE panel per figure, not all three.
trait_annotation <- function(s, rn, traits, show_leg = FALSE) {
  df <- ann_of(s, rn, traits)
  rowAnnotation(df = df, col = TRAIT_COLS[names(df)],
                annotation_name_gp = gpar(fontsize = 5), show_legend = show_leg,
                annotation_legend_param = list(labels_gp = gpar(fontsize = 6),
                                               title_gp  = gpar(fontsize = 7),
                                               grid_height = unit(2.5, "mm"),
                                               grid_width  = unit(2.5, "mm")))
}

# A standalone legend, built from the SAME TRAIT_COLS object the panels use, so
# it cannot drift from what is drawn. The 2x3 arms figure has no room for 15
# stacked legends, and a clipped legend is worse than none.
trait_legend_figure <- function() {
  lgds <- lapply(seq_len(nrow(spec)), function(i) {
    src <- spec$source_column[i]; cm <- TRAIT_COLS[[src]]
    ttl <- sprintf("%s  (%s)", spec$name[i], spec$group[i])
    if (is.function(cm))
      Legend(col_fun = cm, title = ttl, title_gp = gpar(fontsize = 8, fontface = "bold"),
             labels_gp = gpar(fontsize = 7), legend_height = unit(22, "mm"))
    else
      Legend(at = names(cm), legend_gp = gpar(fill = unname(cm)), title = ttl,
             title_gp = gpar(fontsize = 8, fontface = "bold"),
             labels_gp = gpar(fontsize = 7),
             grid_height = unit(4, "mm"), grid_width = unit(4, "mm"))
  })
  # max_height is what makes packLegend wrap into columns; ncol alone stacks.
  packLegend(list = lgds, direction = "vertical", max_height = unit(15, "cm"),
             row_gap = unit(4, "mm"), column_gap = unit(10, "mm"))
}
png(art("fig14_trait_legend.png"), width = 2600, height = 1200, res = 170)
grid.newpage(); draw(trait_legend_figure())
invisible(dev.off())
cat("wrote", basename(art("fig14_trait_legend.png")),
    "-- one legend, shared by every panel in this notebook\n")

ann_of <- function(s, rn, traits){
  m <- W[[s]]$meta[rn, , drop=FALSE]
  cols <- intersect(spec$source_column[match(traits, spec$name)], names(m))
  m[, cols, drop=FALSE]
}

KM_K <- 4                         # k-means at k = 4, to match the 4-group dendrogram cut
i_km <- match(KM_K, c(3, 4, 5))   # step 13 computed k = 3, 4, 5

arm_panel <- function(nm, arm, show_leg = FALSE) {
  r <- RES[[nm]]; d <- r$d; Z <- r$Z
  # Both arms are drawn identically: same distance, same linkage, trees rebuilt
  # within each slice. ONLY the split differs, so any difference in the picture
  # is the clustering method and nothing else.
  if (arm == "hclust") {
    rs <- factor(paste0("P", cutree(r$hclust$hr, KM_K)))
    cs <- factor(paste0("M", cutree(r$hclust$hc, KM_K)))
  } else {
    rs <- r$kmeans$rows[[i_km]]; cs <- r$kmeans$cols[[i_km]]
  }
  Heatmap(Z, name = "z-score", col = colorRamp2(c(-2,0,2), c("#2166AC","white","#B2182B")),
    row_split = rs, column_split = cs,
    cluster_rows = TRUE, cluster_columns = TRUE,
    clustering_distance_rows = dmink,    clustering_method_rows = LINK,
    clustering_distance_columns = dmink, clustering_method_columns = LINK,
    cluster_row_slices = FALSE, cluster_column_slices = FALSE,
    show_row_dend = TRUE,  row_dend_side = "left",  row_dend_width = unit(12, "mm"),
    show_row_names = TRUE, row_names_side = "left", row_names_gp = gpar(fontsize = 3),
    show_column_dend = TRUE, column_dend_side = "top", column_dend_height = unit(12, "mm"),
    show_column_names = FALSE,
    row_title_gp = gpar(fontsize = 8), column_title_gp = gpar(fontsize = 8),
    left_annotation = trait_annotation(d$cohort, rownames(Z), d$traits, show_leg),
    heatmap_legend_param = list(labels_gp = gpar(fontsize = 7), title_gp = gpar(fontsize = 8)))
}

for (cond in c("all15", "varsel", "union")) {
  png(art("fig15_arms_%s.png", cond), width = 3600, height = 2400, res = 150)
  grid.newpage(); pushViewport(viewport(layout = grid.layout(2, 3)))
  for (a in seq_along(c("hclust", "kmeans"))) {
    arm <- c("hclust", "kmeans")[a]
    for (i in seq_along(SITES)) {
      nm <- paste(SITES[i], cond)
      pushViewport(viewport(layout.pos.row = a, layout.pos.col = i))
      sz <- if (arm == "hclust") paste(sort(table(cutree(RES[[nm]]$hclust$hr, KM_K))), collapse = "/")
            else paste(sort(table(RES[[nm]]$kmeans$rows[[i_km]])), collapse = "/")
      # no inline trait legend here -- 15 of them do not fit a 2x3 grid and get
      # clipped. See fig14_trait_legend.png, built from the same colour maps.
      draw(arm_panel(nm, arm, show_leg = FALSE), newpage = FALSE,
        column_title = sprintf("cohort %s -- %s, k = %d -- patients %s",
                               SITES[i], arm, KM_K, sz),
        column_title_gp = gpar(fontsize = 10, fontface = "bold"))
      popViewport()
    }
  }
  popViewport(); invisible(dev.off())
  cat("wrote", basename(art("fig15_arms_%s.png", cond)), "\n")
}

cat("\nARI of the free PROTEIN clustering against the WGCNA module labels (from step 13):\n")
print(K$tab[K$tab$k == KM_K, c("cohort","cond","arm","patient_sizes","ARI_vs_wgcna")],
      row.names = FALSE)

### Reading the two arms

Where the two arms put the same patients together, the block structure is a property of the data
rather than of the algorithm.

The ARI column is the quantitative form of the same question, and it says the modules largely do
**not** break under either arm: cohort A reaches 0.80, cohort C's `union` hierarchical cut reaches
0.886. Cohort B sits at 0.000 throughout — arithmetic, not a finding, since B's panel is a single
module and the label vector is constant.